
# Proper Stacking Ensemble with Out-of-Fold Predictions(Time Series)

- Base model 1: **CatBoost**
- Base model 2: **LightGBM**
- Meta-model: **Multinomial Logistic Regression**
- CV style: **expanding / rolling time-series folds**
- Key fix: the meta-model is trained **only on rows that actually received OOF predictions**

## Why the earlier error happened
In rolling time-series CV, the earliest historical chunk is often used **only for training**, not validation.  
That means some training rows **should remain uncovered** by OOF predictions.  
So instead of asserting that **all** rows must have OOF predictions, we now:

1. track which rows were covered by validation folds,
2. verify there are **no NaNs among covered rows**, and
3. train the meta-model only on those covered rows.

## Expected outputs
This notebook saves:
- `../models/stacking_oof_cv_summary.csv`
- `../models/stacking_final_results.csv`
- `../models/stacking_meta_model.joblib`
- `../models/stacking_base_cb.cbm`
- `../models/stacking_base_lgbm.joblib`


In [28]:

from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    average_precision_score,
    mean_squared_error
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

import joblib

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SUPERVISED_PATH = DATA_DIR / "supervised_hood_3h_multiclass.csv"
SEL_FEATURES_PATH = DATA_DIR / "logreg_selected_top25_features.txt"

OUT_CV_SUMMARY = MODEL_DIR / "stacking_oof_cv_summary.csv"
OUT_FINAL_RESULTS = MODEL_DIR / "stacking_oof_final_summary.csv"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

N_CLASSES = 3
USE_SELECTED_FEATURES = False   # set True if you want to use the saved top-feature list
N_TIME_CHUNKS = 5               # 5 chunks -> 4 expanding folds
MIN_TRAIN_CHUNKS = 1




In [29]:

# ------------------------------------------------------------
# Load supervised dataset
# ------------------------------------------------------------
df = pd.read_csv(SUPERVISED_PATH, low_memory=False)

df["time_3h"] = pd.to_datetime(df["time_3h"], errors="coerce")
df["HOOD_158_CODE"] = df["HOOD_158_CODE"].astype(str).str.zfill(3)
df["y_class"] = pd.to_numeric(df["y_class"], errors="coerce").astype("int8")

df = df.sort_values(["time_3h", "HOOD_158_CODE"]).reset_index(drop=True)

print("Shape:", df.shape)
print("\nClass distribution:")
print(df["y_class"].value_counts(normalize=True).sort_index().round(4))

display(df.head())


Shape: (1383922, 50)

Class distribution:
y_class
0    0.8930
1    0.0927
2    0.0143
Name: proportion, dtype: float64


,HOOD_158_CODE,time_3h,collisions,injury_collisions,ftr_collisions,pd_collisions,pedestrian_collisions,bicycle_collisions,pressure_sea,wind_speed,...,visibility_lag_1,rain_lag_1,snow_lag_1,snow_on_ground_lag_1,wind_speed_lag_1,relative_humidity_lag_1,pressure_sea_lag_1,cloud_cover_8_lag_1,y_count_next,y_class
0,001,2023-01-02,0,0,0,0,0,0,101.643,8.0,...,9700.0,8.4,0.0,0.0,11.333,100.0,101.577,7.333,0.0,0
1,002,2023-01-02,0,0,0,0,0,0,101.643,8.0,...,9700.0,8.4,0.0,0.0,11.333,100.0,101.577,7.333,0.0,0
2,003,2023-01-02,0,0,0,0,0,0,101.643,8.0,...,9700.0,8.4,0.0,0.0,11.333,100.0,101.577,7.333,0.0,0
3,004,2023-01-02,0,0,0,0,0,0,101.643,8.0,...,9700.0,8.4,0.0,0.0,11.333,100.0,101.577,7.333,1.0,1
4,005,2023-01-02,0,0,0,0,0,0,101.643,8.0,...,9700.0,8.4,0.0,0.0,11.333,100.0,101.577,7.333,0.0,0


In [30]:

# ------------------------------------------------------------
# Time-based split
# Train: <= 2024-12-31
# Val:   2025-01-01 to 2025-06-30
# Test:  >= 2025-07-01
# ------------------------------------------------------------
t = df["time_3h"]

train_end = pd.Timestamp("2024-12-31 23:59:59")
val_end   = pd.Timestamp("2025-06-30 23:59:59")

train_mask = t <= train_end
val_mask   = (t > train_end) & (t <= val_end)
test_mask  = t > val_end

y = df["y_class"].copy()
X = df.drop(columns=[c for c in ["y_class", "y_count_next"] if c in df.columns], errors="ignore").copy()
X = X.drop(columns=["time_3h"], errors="ignore")

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_val = X.loc[val_mask].copy()
y_val = y.loc[val_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

time_train = t.loc[train_mask].reset_index(drop=True)
time_val = t.loc[val_mask].reset_index(drop=True)
time_test = t.loc[test_mask].reset_index(drop=True)

# Reset train/val/test indices to 0..n-1 for safe OOF indexing
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("Train shape:", X_train.shape)
print("Val shape:  ", X_val.shape)
print("Test shape: ", X_test.shape)


Train shape: (922720, 47)
Val shape:   (228784, 47)
Test shape:  (232418, 47)


In [31]:

# ------------------------------------------------------------
# Optional feature selection
# ------------------------------------------------------------
if USE_SELECTED_FEATURES and SEL_FEATURES_PATH.exists():
    with open(SEL_FEATURES_PATH, "r") as f:
        selected = [line.strip() for line in f if line.strip()]
    selected = [c for c in selected if c in X_train.columns]
    if "HOOD_158_CODE" in X_train.columns and "HOOD_158_CODE" not in selected:
        selected.append("HOOD_158_CODE")
    print("Using selected feature list:", len(selected))
else:
    selected = list(X_train.columns)
    print("Using all features:", len(selected))

X_train = X_train[selected].copy()
X_val   = X_val[selected].copy()
X_test  = X_test[selected].copy()

cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
if "HOOD_158_CODE" in X_train.columns and "HOOD_158_CODE" not in cat_cols:
    cat_cols.append("HOOD_158_CODE")

num_cols = [c for c in X_train.columns if c not in cat_cols]

# Make categorical columns safe for CatBoost
for frame in [X_train, X_val, X_test]:
    for c in cat_cols:
        frame[c] = frame[c].astype(str).fillna("missing")

print("Categorical columns:", len(cat_cols))
print("Numeric columns:", len(num_cols))
print("\nFirst few categorical columns:", cat_cols[:10])


Using all features: 47
Categorical columns: 1
Numeric columns: 46

First few categorical columns: ['HOOD_158_CODE']


In [32]:
# ------------------------------------------------------------
# Evaluation helpers
# ------------------------------------------------------------
def plot_confusion(cm, labels=(0, 1, 2), title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    plt.show()


def eval_multiclass(y_true, y_pred, proba=None, name="Model", split="test"):
    """
    Evaluate a 3-class classifier with the compact metric set used in the final project:
    - macro_recall
    - recall_2
    - precision_2
    - ap_class2
    - macro_map
    - prob_rmse_macro

    Notes
    -----
    * `pred_class2_rate` is kept as a diagnostic column for model behavior.
    * Confusion-matrix cells are kept for debugging / optional appendix use.
    """
    y_true_arr = np.asarray(y_true).astype(int)
    y_pred_arr = np.asarray(y_pred).astype(int)

    cm = confusion_matrix(y_true_arr, y_pred_arr, labels=[0, 1, 2])
    rec = recall_score(y_true_arr, y_pred_arr, labels=[0, 1, 2], average=None, zero_division=0)
    macro_rec = recall_score(y_true_arr, y_pred_arr, average="macro", zero_division=0)
    prec2 = precision_score((y_true_arr == 2).astype(int), (y_pred_arr == 2).astype(int), zero_division=0)
    pred_class2_rate = float((y_pred_arr == 2).mean())

    out = {
        "model": name,
        "split": split,
        "macro_recall": float(macro_rec),
        "recall_0": float(rec[0]),
        "recall_1": float(rec[1]),
        "recall_2": float(rec[2]),
        "precision_2": float(prec2),
        "pred_class2_rate": pred_class2_rate,
        "cm_00": int(cm[0, 0]), "cm_01": int(cm[0, 1]), "cm_02": int(cm[0, 2]),
        "cm_10": int(cm[1, 0]), "cm_11": int(cm[1, 1]), "cm_12": int(cm[1, 2]),
        "cm_20": int(cm[2, 0]), "cm_21": int(cm[2, 1]), "cm_22": int(cm[2, 2]),
    }

    if proba is None:
        out["ap_class0"] = np.nan
        out["ap_class1"] = np.nan
        out["ap_class2"] = np.nan
        out["macro_map"] = np.nan
        out["prob_rmse_macro"] = np.nan
        return out

    proba_arr = np.asarray(proba, dtype=float)
    ap_vals = []
    prob_rmse_vals = []

    for k in range(3):
        try:
            ap_k = average_precision_score((y_true_arr == k).astype(int), proba_arr[:, k])
        except Exception:
            ap_k = np.nan
        ap_vals.append(ap_k)

        try:
            yk = (y_true_arr == k).astype(float)
            rmse_k = float(np.sqrt(mean_squared_error(yk, proba_arr[:, k])))
        except Exception:
            rmse_k = np.nan
        prob_rmse_vals.append(rmse_k)

    out["ap_class0"] = float(ap_vals[0]) if pd.notna(ap_vals[0]) else np.nan
    out["ap_class1"] = float(ap_vals[1]) if pd.notna(ap_vals[1]) else np.nan
    out["ap_class2"] = float(ap_vals[2]) if pd.notna(ap_vals[2]) else np.nan
    out["macro_map"] = float(np.nanmean(ap_vals))
    out["prob_rmse_macro"] = float(np.nanmean(prob_rmse_vals))
    return out

In [33]:

# ------------------------------------------------------------
# Class weights from the training set
# ------------------------------------------------------------
class_counts = y_train.value_counts().sort_index()
total = class_counts.sum()

# sklearn-style balanced weights
weights = (total / (N_CLASSES * class_counts)).to_dict()

print("Class counts:")
print(class_counts)
print("\nClass weights:")
print(weights)


Class counts:
y_class
0    823183
1     86350
2     13187
Name: count, dtype: int64

Class weights:
{0: 0.373639073369267, 1: 3.561937849835939, 2: 23.323980688051364}



## Build expanding time-series folds inside the training period

We split the **training period only** by unique `time_3h` values into contiguous chunks.  
For each fold:

- training = all earlier chunks
- validation = the next chunk

This avoids future leakage and gives proper OOF predictions for stacking.


In [34]:

# ------------------------------------------------------------
# Time-series fold builder
# ------------------------------------------------------------
def make_expanding_time_folds(time_series, n_chunks=5, min_train_chunks=1):
    unique_times = np.array(sorted(pd.Series(time_series).dropna().unique()))
    time_chunks = np.array_split(unique_times, n_chunks)

    folds = []
    for i in range(min_train_chunks, len(time_chunks)):
        train_times = np.concatenate(time_chunks[:i])
        val_times = time_chunks[i]

        tr_idx = np.where(pd.Series(time_series).isin(train_times).values)[0]
        va_idx = np.where(pd.Series(time_series).isin(val_times).values)[0]

        if len(tr_idx) == 0 or len(va_idx) == 0:
            continue

        folds.append((tr_idx, va_idx))

    return folds, time_chunks

folds, time_chunks = make_expanding_time_folds(time_train, n_chunks=N_TIME_CHUNKS, min_train_chunks=MIN_TRAIN_CHUNKS)

print("Number of unique training times:", len(pd.Series(time_train).dropna().unique()))
print("Time chunks:", len(time_chunks))
print("Folds:", len(folds))

for i, (tr_idx, va_idx) in enumerate(folds, start=1):
    print(
        f"Fold {i}: "
        f"train_rows={len(tr_idx):,}, val_rows={len(va_idx):,}, "
        f"train_time=[{time_train.iloc[tr_idx].min()} -> {time_train.iloc[tr_idx].max()}], "
        f"val_time=[{time_train.iloc[va_idx].min()} -> {time_train.iloc[va_idx].max()}]"
    )


Number of unique training times: 5840
Time chunks: 5
Folds: 4
Fold 1: train_rows=184,544, val_rows=184,544, train_time=[2023-01-02 00:00:00 -> 2023-05-27 21:00:00], val_time=[2023-05-28 00:00:00 -> 2023-10-20 21:00:00]
Fold 2: train_rows=369,088, val_rows=184,544, train_time=[2023-01-02 00:00:00 -> 2023-10-20 21:00:00], val_time=[2023-10-21 00:00:00 -> 2024-03-14 21:00:00]
Fold 3: train_rows=553,632, val_rows=184,544, train_time=[2023-01-02 00:00:00 -> 2024-03-14 21:00:00], val_time=[2024-03-15 00:00:00 -> 2024-08-07 21:00:00]
Fold 4: train_rows=738,176, val_rows=184,544, train_time=[2023-01-02 00:00:00 -> 2024-08-07 21:00:00], val_time=[2024-08-08 00:00:00 -> 2024-12-31 21:00:00]


In [35]:

# ------------------------------------------------------------
# Model builders
# ------------------------------------------------------------
try:
    from catboost import CatBoostClassifier
except Exception as e:
    raise RuntimeError(f"CatBoost is not available. Install it first. Error: {e}")

try:
    from lightgbm import LGBMClassifier
except Exception as e:
    raise RuntimeError(f"LightGBM is not available. Install it first. Error: {e}")

cat_feature_idx = [X_train.columns.get_loc(c) for c in cat_cols]

def build_cb_model(seed):
    return CatBoostClassifier(
        loss_function="MultiClassOneVsAll",
        iterations=1500,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=10.0,
        random_seed=seed,
        verbose=False,
        auto_class_weights="SqrtBalanced",
        od_type="Iter",
        od_wait=200
    )

def build_lgbm_pipe(seed):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols),
        ],
        remainder="drop"
    )

    clf = LGBMClassifier(
        objective="multiclass",
        num_class=N_CLASSES,
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        class_weight=weights,
        n_jobs=-1,
        verbose=-1
    )

    return Pipeline(steps=[
        ("preprocess", preprocess),
        ("clf", clf)
    ])


In [36]:

# ============================================================
# OOF predictions for proper stacking
# IMPORTANT FIX:
# In time-series CV, not every row must be covered by validation.
# We only require that covered rows contain no NaNs.
# ============================================================
n_train = len(X_train)

oof_cb = np.full((n_train, N_CLASSES), np.nan, dtype=float)
oof_lgbm = np.full((n_train, N_CLASSES), np.nan, dtype=float)
oof_mask = np.zeros(n_train, dtype=bool)

cv_rows = []
cb_tree_counts = []

for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
    X_tr_fold = X_train.iloc[tr_idx].copy()
    y_tr_fold = y_train.iloc[tr_idx].copy()

    X_va_fold = X_train.iloc[va_idx].copy()
    y_va_fold = y_train.iloc[va_idx].copy()

    # -----------------------------
    # CatBoost
    # -----------------------------
    cb_model = build_cb_model(RANDOM_SEED + fold_id)
    cb_model.fit(
        X_tr_fold, y_tr_fold,
        cat_features=cat_feature_idx,
        eval_set=(X_va_fold, y_va_fold),
        use_best_model=True
    )

    proba_cb = np.asarray(cb_model.predict_proba(X_va_fold), dtype=float)
    oof_cb[va_idx, :] = proba_cb
    cb_tree_counts.append(cb_model.tree_count_)

    pred_cb = np.argmax(proba_cb, axis=1)
    m_cb = eval_multiclass(y_va_fold, pred_cb, proba_cb, name=f"CatBoost Fold {fold_id}")
    m_cb["fold"] = fold_id
    m_cb["base_model"] = "CatBoost"
    cv_rows.append(m_cb)

    # -----------------------------
    # LightGBM
    # -----------------------------
    lgbm_model = build_lgbm_pipe(RANDOM_SEED + fold_id)
    lgbm_model.fit(X_tr_fold, y_tr_fold)

    proba_lgbm = np.asarray(lgbm_model.predict_proba(X_va_fold), dtype=float)
    oof_lgbm[va_idx, :] = proba_lgbm

    pred_lgbm = np.argmax(proba_lgbm, axis=1)
    m_lgbm = eval_multiclass(y_va_fold, pred_lgbm, proba_lgbm, name=f"LightGBM Fold {fold_id}")
    m_lgbm["fold"] = fold_id
    m_lgbm["base_model"] = "LightGBM"
    cv_rows.append(m_lgbm)

    # mark which rows actually received OOF predictions
    oof_mask[va_idx] = True

    print(
        f"Fold {fold_id}/{len(folds)} done | "
        f"CatBoost macro={m_cb['macro_recall']:.4f} | "
        f"LightGBM macro={m_lgbm['macro_recall']:.4f}"
    )

covered_n = int(oof_mask.sum())
uncovered_n = int((~oof_mask).sum())

print("\nOOF-covered rows:", f"{covered_n:,}")
print("OOF-uncovered rows:", f"{uncovered_n:,}")

assert covered_n > 0, "No rows received OOF predictions. Check your fold construction."
assert not np.isnan(oof_cb[oof_mask]).any(), "CatBoost covered OOF predictions contain NaN."
assert not np.isnan(oof_lgbm[oof_mask]).any(), "LightGBM covered OOF predictions contain NaN."

print("Median CatBoost tree_count across folds:", int(np.median(cb_tree_counts)))

cv_df = pd.DataFrame(cv_rows)

rate_col = "pred_class2_rate" if "pred_class2_rate" in cv_df.columns else (
    "pred2_rate" if "pred2_rate" in cv_df.columns else None
)

preferred_cols = [
    "fold",
    "base_model",
    "macro_recall",
    "recall_0",
    "recall_1",
    "recall_2",
    "precision_2",
    "ap_class2",
    "macro_map",
    "prob_rmse_macro",
]

if rate_col is not None:
    preferred_cols.insert(7, rate_col)

display_cols = [c for c in preferred_cols if c in cv_df.columns]

display(cv_df[display_cols].head())


Fold 1/4 done | CatBoost macro=0.4177 | LightGBM macro=0.4616
Fold 2/4 done | CatBoost macro=0.4053 | LightGBM macro=0.4767
Fold 3/4 done | CatBoost macro=0.4276 | LightGBM macro=0.5239
Fold 4/4 done | CatBoost macro=0.4317 | LightGBM macro=0.5326

OOF-covered rows: 738,176
OOF-uncovered rows: 184,544
Median CatBoost tree_count across folds: 549


,fold,base_model,macro_recall,recall_0,recall_1,recall_2,precision_2,pred_class2_rate,ap_class2,macro_map,prob_rmse_macro
0,1,CatBoost,0.417684,0.957858,0.148056,0.147137,0.195861,0.010735,0.112695,0.423616,0.264139
1,1,LightGBM,0.461576,0.740639,0.451447,0.192643,0.078943,0.034870,0.066074,0.392392,0.335520
2,2,CatBoost,0.405307,0.966534,0.115573,0.133813,0.201484,0.009494,0.111654,0.420862,0.256666
3,2,LightGBM,0.476671,0.744831,0.440678,0.244503,0.079365,0.044038,0.073925,0.394494,0.342374
4,3,CatBoost,0.427621,0.958278,0.148607,0.175979,0.214253,0.011710,0.136348,0.433829,0.261068


In [37]:
# ------------------------------------------------------------
# CV summary by base model
# ------------------------------------------------------------
cv_summary = (
    cv_df.groupby("base_model", as_index=False)[
        [
            "macro_recall",
            "recall_0",
            "recall_1",
            "recall_2",
            "precision_2",
            "pred_class2_rate",
            "ap_class2",
            "macro_map",
            "prob_rmse_macro",
        ]
    ]
    .mean()
    .sort_values(["macro_recall", "ap_class2", "recall_2"], ascending=False)
    .reset_index(drop=True)
)

print("Base-model CV summary:")
display(cv_summary)

Base-model CV summary:


,base_model,macro_recall,recall_0,recall_1,recall_2,precision_2,pred_class2_rate,ap_class2,macro_map,prob_rmse_macro
0,LightGBM,0.498709,0.712232,0.451272,0.332622,0.082392,0.058894,0.088889,0.398617,0.351434
1,CatBoost,0.420566,0.958867,0.142437,0.160394,0.203912,0.011530,0.121748,0.427212,0.262031


In [38]:

# ------------------------------------------------------------
# Build meta-training data from covered OOF rows only
# ------------------------------------------------------------
def make_meta_X(p_cb, p_lgbm):
    return np.hstack([p_cb, p_lgbm])

Xmeta_train = make_meta_X(oof_cb[oof_mask], oof_lgbm[oof_mask])
ymeta_train = y_train.iloc[oof_mask].copy()

print("Meta-train shape:", Xmeta_train.shape, ymeta_train.shape)
print("Covered ratio:", round(float(oof_mask.mean()), 4))


Meta-train shape: (738176, 6) (738176,)
Covered ratio: 0.8


In [40]:

# ------------------------------------------------------------
# Fit final base models on the full training set
# ------------------------------------------------------------
cb_final = build_cb_model(RANDOM_SEED)
cb_final.fit(
    X_train, y_train,
    cat_features=cat_feature_idx,
    eval_set=(X_val, y_val),
    use_best_model=True
)

lgbm_final = build_lgbm_pipe(RANDOM_SEED)
lgbm_final.fit(X_train, y_train)

# Base-model probabilities for val/test
proba_val_cb = np.asarray(cb_final.predict_proba(X_val), dtype=float)
proba_val_lgbm = np.asarray(lgbm_final.predict_proba(X_val), dtype=float)
Xmeta_val = make_meta_X(proba_val_cb, proba_val_lgbm)

proba_test_cb = np.asarray(cb_final.predict_proba(X_test), dtype=float)
proba_test_lgbm = np.asarray(lgbm_final.predict_proba(X_test), dtype=float)
Xmeta_test = make_meta_X(proba_test_cb, proba_test_lgbm)

# Optional: also evaluate base models on test for comparison
pred_test_cb = np.argmax(proba_test_cb, axis=1)
pred_test_lgbm = np.argmax(proba_test_lgbm, axis=1)

base_test_rows = [
    eval_multiclass(y_test, pred_test_cb, proba_test_cb, name="CatBoost Argmax - Test"),
    eval_multiclass(y_test, pred_test_lgbm, proba_test_lgbm, name="LightGBM Argmax - Test"),
]

base_test_df = pd.DataFrame(base_test_rows)

base_test_display_cols = [
    c for c in [
        "model",
        "split",
        "macro_recall",
        "recall_2",
        "precision_2",
        "pred_class2_rate",
        "ap_class2",
        "macro_map",
        "prob_rmse_macro",
    ]
    if c in base_test_df.columns
]

display(base_test_df[base_test_display_cols])

,model,split,macro_recall,recall_2,precision_2,pred_class2_rate,ap_class2,macro_map,prob_rmse_macro
0,CatBoost Argmax - Test,test,0.436124,0.201407,0.173792,0.017008,0.123335,0.428370,0.265187
1,LightGBM Argmax - Test,test,0.535660,0.497508,0.076923,0.094919,0.111032,0.402844,0.372717


In [41]:
# ------------------------------------------------------------
# Meta model
# We keep the meta-model light and stable.
# Because stacking is already the key improvement, we avoid extra complexity here.
# ------------------------------------------------------------
meta = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        multi_class="multinomial",
        class_weight="balanced",
        random_state=RANDOM_SEED
    ))
])

meta.fit(Xmeta_train, ymeta_train)

# Validation / Test predictions
pred_val_meta = meta.predict(Xmeta_val)
pred_test_meta = meta.predict(Xmeta_test)

proba_val_meta = meta.predict_proba(Xmeta_val)
proba_test_meta = meta.predict_proba(Xmeta_test)

metrics_val_meta = eval_multiclass(
    y_val, pred_val_meta, proba_val_meta,
    name="Proper Stacking Meta - Val", split="val"
)
metrics_test_meta = eval_multiclass(
    y_test, pred_test_meta, proba_test_meta,
    name="Proper Stacking Meta - Test", split="test"
)

final_results = pd.DataFrame(base_test_rows + [metrics_val_meta, metrics_test_meta])

final_display_cols = [
    c for c in [
        "model",
        "split",
        "macro_recall",
        "recall_2",
        "precision_2",
        "pred_class2_rate",
        "ap_class2",
        "macro_map",
        "prob_rmse_macro",
    ]
    if c in final_results.columns
]

display(final_results[final_display_cols].sort_values(
    ["macro_recall", "ap_class2", "recall_2"],
    ascending=False
).reset_index(drop=True))

,model,split,macro_recall,recall_2,precision_2,pred_class2_rate,ap_class2,macro_map,prob_rmse_macro
0,Proper Stacking Meta - Test,test,0.551506,0.666080,0.065993,0.148130,0.128423,0.405670,0.398418
1,Proper Stacking Meta - Val,val,0.545886,0.621329,0.065466,0.134201,0.111376,0.401692,0.390237
2,LightGBM Argmax - Test,test,0.535660,0.497508,0.076923,0.094919,0.111032,0.402844,0.372717
3,CatBoost Argmax - Test,test,0.436124,0.201407,0.173792,0.017008,0.123335,0.428370,0.265187


In [42]:
# ------------------------------------------------------------
# Save compact comparison outputs only
# Notebook 12 is for comparison, not dashboard deployment.
# So we save only the compact summary files needed by Notebook 15.
# ------------------------------------------------------------
if "final_results" not in globals():
    raise RuntimeError(
        "final_results is not defined. Run the previous Meta model cell first, "
        "then rerun this Save artifacts cell."
    )

if "cv_summary" not in globals():
    raise RuntimeError(
        "cv_summary is not defined. Run the CV summary cell first, "
        "then rerun this Save artifacts cell."
    )

# Final summary for Notebook 15 comparison
final_summary = final_results[
    [c for c in [
        "model",
        "split",
        "macro_recall",
        "recall_2",
        "precision_2",
        "ap_class2",
        "macro_map",
        "prob_rmse_macro"
    ] if c in final_results.columns]
].copy()
final_summary.to_csv(OUT_FINAL_RESULTS, index=False)

# CV summary for Notebook 15 comparison
cv_summary_to_save = cv_summary.rename(columns={"base_model": "model"}).copy()
cv_summary_to_save = cv_summary_to_save[
    [c for c in [
        "model",
        "macro_recall",
        "recall_2",
        "precision_2",
        "ap_class2",
        "macro_map",
        "prob_rmse_macro"
    ] if c in cv_summary_to_save.columns]
].copy()
cv_summary_to_save.to_csv(OUT_CV_SUMMARY, index=False)

print("Saved:", OUT_CV_SUMMARY)
print("Saved:", OUT_FINAL_RESULTS)

display(final_summary.sort_values(
    ["macro_recall", "ap_class2", "recall_2"],
    ascending=False
).reset_index(drop=True))

Saved: ../models/stacking_oof_cv_summary.csv
Saved: ../models/stacking_oof_final_summary.csv


,model,split,macro_recall,recall_2,precision_2,ap_class2,macro_map,prob_rmse_macro
0,Proper Stacking Meta - Test,test,0.551506,0.666080,0.065993,0.128423,0.405670,0.398418
1,Proper Stacking Meta - Val,val,0.545886,0.621329,0.065466,0.111376,0.401692,0.390237
2,LightGBM Argmax - Test,test,0.535660,0.497508,0.076923,0.111032,0.402844,0.372717
3,CatBoost Argmax - Test,test,0.436124,0.201407,0.173792,0.123335,0.428370,0.265187
